[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/10_decision_trees/90_exercises.ipynb)

# Bài tập — Chương 10: Decision Trees (L0 → L7)

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/10_decision_trees

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.datasets import make_classification_2d
from ml_course.utils import repo_path

rng = np.random.default_rng(0)

## L0 — Nhớ khái niệm

**Câu 1.** Viết công thức entropy, Gini, information gain. Liệt kê 4 hyperparameter pre-pruning và công thức chi phí của cost-complexity pruning.

✍️ Trả lời:

## L1 — Giải thích bằng lời

**Câu 2.** Một đồng nghiệp nói: "Cây quyết định dễ diễn giải nên luôn là lựa chọn tốt cho bài toán cần giải thích". Phản biện với ít nhất 3 ý.

✍️ Trả lời:

## L2 — Tính tay

**Câu 3.** Node cha có 10 mẫu: 6 lớp A, 4 lớp B. Một split chia thành trái (5 A, 1 B) và phải (1 A, 3 B). Tính entropy cha, entropy hai con, information gain; Gini cha và giảm Gini.

In [ ]:
### START CODE HERE ### (≈ 6 dòng)
H = None
G = None
h_parent = None
info_gain = None
g_parent = None
gini_decrease = None
### END CODE HERE ###

In [ ]:
check_close(h_parent, 0.970951, tol=1e-5, message="entropy cha")
check_close(info_gain, 0.256, tol=1e-3, message="information gain")
check_close(g_parent, 0.48, message="Gini cha")
check_close(gini_decrease, 0.1633, tol=1e-3, message="giảm Gini")

**Câu 4.** Regression tree, node có $y = [1, 2, 3, 10, 11, 12]$. So sánh hai split: (a) $\{1, 2, 3\}$ | $\{10, 11, 12\}$; (b) $\{1, 2, 3, 10\}$ | $\{11, 12\}$. Tính mức **giảm phương sai** của mỗi split và dự đoán ở hai lá của split tốt hơn.

In [ ]:
y_node = np.array([1, 2, 3, 10, 11, 12.0])
### START CODE HERE ### (≈ 4 dòng)
var_drop = None
drop_a = None
drop_b = None
leaf_values = None
### END CODE HERE ###

In [ ]:
check_close(drop_a, 20.25, message="giảm phương sai split (a)")
check_close(drop_b, 12.5, tol=1e-6, message="giảm phương sai split (b)")
check_close(leaf_values, (2.0, 11.0), message="dự đoán ở hai lá")

## L3 — Code

**Bài 5.** Viết `best_split(X, y)` cho **phân loại** (Gini) trên **mọi feature**: trả về `(feature, threshold, gain)`. Dùng sắp xếp + `cumsum` (không vòng lặp qua ngưỡng). Ngưỡng là trung điểm giữa hai giá trị liên tiếp khác nhau.

In [ ]:
def best_split(X, y):
    n, d = X.shape
    classes = np.unique(y)
    onehot = (y[:, None] == classes[None, :]).astype(float)
    parent = 1 - ((onehot.mean(0)) ** 2).sum()
    best = (None, None, 0.0)
    for j in range(d):
        order = np.argsort(X[:, j], kind="mergesort")
        xs, oh = X[order, j], onehot[order]
        ### START CODE HERE ### (≈ 9 dòng)
        left = None        # (n-1, C) số đếm lớp bên trái tại mỗi vị trí cắt — dùng np.cumsum
        right = None       # số đếm bên phải = tổng − left
        n_left = None      # (n-1, 1)
        n_right = None
        child = None       # Gini có trọng số của hai con; đặt np.inf tại vị trí xs[i] == xs[i+1]
        i = None           # vị trí child nhỏ nhất
        # nếu parent - child[i] > best[2]: best = (j, trung điểm xs[i], xs[i+1], gain)
        ### END CODE HERE ###
    return best

In [ ]:
Xs, ys = make_classification_2d("blobs", n=400, noise=0.4, seed=5)
f, t, g = best_split(Xs, ys)
stump = DecisionTreeClassifier(max_depth=1).fit(Xs, ys)
check_equal(f, int(stump.tree_.feature[0]), "feature giống sklearn")
check_close(t, stump.tree_.threshold[0], tol=1e-6, message="ngưỡng giống sklearn")
imp = stump.tree_.impurity
w = stump.tree_.weighted_n_node_samples
check_close(g, imp[0] - (w[1] * imp[1] + w[2] * imp[2]) / w[0], tol=1e-9, message="gain giống sklearn")

**Bài 6.** Dùng `best_split` viết hàm đệ quy `build_tree(X, y, depth, max_depth)` trả về dict lồng nhau (`{"leaf": lớp}` hoặc `{"feature", "threshold", "left", "right"}`) và `predict_tree(node, x)`. Đạt accuracy giống `DecisionTreeClassifier(max_depth=4)` trên moons.

In [ ]:
def build_tree(X, y, depth=0, max_depth=4):
    values, counts = np.unique(y, return_counts=True)
    majority = values[np.argmax(counts)]
    if depth >= max_depth or len(values) == 1:
        return {"leaf": majority}
    ### START CODE HERE ### (≈ 6 dòng)
    f, t, g = None, None, None    # best_split(X, y); nếu f is None → trả về lá
    mask = None                   # X[:, f] <= t
    return None                   # dict {"feature", "threshold", "left": build_tree(...), "right": build_tree(...)}
    ### END CODE HERE ###


def predict_tree(node, x):
    ### START CODE HERE ### (≈ 3 dòng)
    # đi xuống trái nếu x[feature] <= threshold, ngược lại sang phải, tới khi gặp {"leaf": ...}
    pass
    ### END CODE HERE ###

In [ ]:
Xm, ym = make_classification_2d("moons", n=500, noise=0.3, seed=8)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(Xm, ym, test_size=0.3, random_state=0)
tree_dict = build_tree(Xm_tr, ym_tr, max_depth=4)
acc_ours = np.mean([predict_tree(tree_dict, x) == t for x, t in zip(Xm_te, ym_te)])
acc_sk = DecisionTreeClassifier(max_depth=4, random_state=0).fit(Xm_tr, ym_tr).score(Xm_te, ym_te)
print(f"scratch = {acc_ours:.4f} | sklearn = {acc_sk:.4f}")
check_close(acc_ours, acc_sk, tol=0.02, message="accuracy ≈ sklearn")

## L4 — Debug

**Bài 7.** Đoạn code dưới đây báo "cây đạt 100% accuracy, model tuyệt vời!". Tìm **2 lỗi**: (1) đánh giá trên chính tập train; (2) cây không giới hạn. Sửa: chọn `max_depth` và `min_samples_leaf` bằng `GridSearchCV` (5-fold) trên train, báo cáo accuracy **test** (`test_acc`) và **train** (`train_acc`) của model tốt nhất.

In [ ]:
Xd, yd = make_classification_2d("moons", n=600, noise=0.4, seed=21)
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.3, random_state=0, stratify=yd)
### START CODE HERE ### (≈ 5 dòng)
model = DecisionTreeClassifier().fit(Xd_tr, yd_tr)
train_acc = model.score(Xd_tr, yd_tr)
test_acc = model.score(Xd_tr, yd_tr)          # 🐛
### END CODE HERE ###
print(f"train = {train_acc:.3f} | test = {test_acc:.3f}")

In [ ]:
check_less(train_acc, 0.999, "không còn thuộc lòng tập train")
check_greater(test_acc, 0.83, "test accuracy > 0.83")

## L5 — Phân tích

**Bài 8.** Đo **độ ổn định** của cây: train `DecisionTreeClassifier(max_depth=d)` trên 30 mẫu bootstrap của dữ liệu moons (n=1000) với $d \in \{2, 4, 8, \text{None}\}$. Với mỗi $d$, tính `disagreement[d]` = tỉ lệ trung bình các điểm trên lưới mà hai cây (trong 30 cây) dự đoán **khác nhau** (lấy trung bình trên mọi cặp, hoặc ước lượng bằng tỉ lệ điểm mà dự đoán khác với "phiếu đa số" của 30 cây). Vẽ và nhận xét quan hệ độ sâu ↔ variance.

In [ ]:
Xv, yv = make_classification_2d("moons", n=1000, noise=0.35, seed=4)
gx, gy = np.meshgrid(np.linspace(-1.5, 2.5, 80), np.linspace(-1.2, 1.7, 80))
G = np.c_[gx.ravel(), gy.ravel()]
disagreement = {}
for d in [2, 4, 8, None]:
    ### START CODE HERE ### (≈ 6 dòng)
    preds = None              # (30, số điểm lưới): dự đoán của 30 cây train trên bootstrap (seed b = 0..29)
    majority = None           # phiếu đa số của 30 cây tại mỗi điểm lưới
    disagreement[d] = None    # tỉ lệ trung bình dự đoán khác phiếu đa số
    ### END CODE HERE ###
print({str(k): round(v, 4) for k, v in disagreement.items()})
plt.bar([str(k) for k in disagreement], list(disagreement.values()))
plt.xlabel("max_depth")
plt.ylabel("tỉ lệ dự đoán khác phiếu đa số")
plt.title("Cây càng sâu càng không ổn định")
plt.show()

In [ ]:
check_greater(disagreement[None], 2 * disagreement[2], "cây không giới hạn kém ổn định hơn nhiều so với depth 2")

✍️ Nhận xét:

## L6 — Thực tế

**Bài 9.** Dự đoán giá nhà (`house_prices.csv`, target `price_billion_vnd`) bằng `DecisionTreeRegressor`: one-hot `district`, chọn `ccp_alpha` bằng CV (dùng `cost_complexity_pruning_path` để lấy ứng viên), báo cáo MAE test (`test_mae`) và số lá. Mục tiêu: `test_mae < 0.75` (tỉ VND). In 5 luật đầu tiên của cây (dùng `export_text` với `max_depth=2`).

In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.tree import export_text

houses = pd.read_csv(repo_path("data", "sample", "house_prices.csv"))
Xh = pd.get_dummies(houses.drop(columns=["price_billion_vnd"]), columns=["district"], dtype=float)
yh = houses["price_billion_vnd"]
Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(Xh, yh, test_size=0.25, random_state=0)
### START CODE HERE ### (≈ 7 dòng)
path = None
candidates = None
search = None
best_tree = None
test_mae = None
### END CODE HERE ###
print(f"ccp_alpha = {search.best_params_['ccp_alpha']:.4g} | số lá = {best_tree.get_n_leaves()} | MAE test = {test_mae:.3f} tỉ")
print(export_text(best_tree, feature_names=list(Xh.columns), max_depth=2))

In [ ]:
check_less(test_mae, 0.75, "MAE test < 0.75 tỉ")

## L7 — Nâng cao

**Bài 10.** Cài đặt **cost-complexity pruning** from scratch cho regression tree của sklearn: với cây đã train (`tree_`), tính cho mỗi node trong $t$: $R(t)$ = SSE nếu biến $t$ thành lá, $R(T_t)$ = tổng SSE các lá của cây con, $|T_t|$ = số lá cây con, và **weakest link** $g(t) = \frac{R(t) - R(T_t)}{|T_t| - 1}$. Trả về $\min_t g(t)$ — chính là $\alpha$ hiệu quả đầu tiên (khác 0) trong `cost_complexity_pruning_path` (sklearn dùng impurity có trọng số $\frac{n_t}{n}\cdot\text{MSE}_t$, tức $\text{SSE}/n$).

In [ ]:
def weakest_link_alpha(fitted_tree):
    T = fitted_tree.tree_
    n_total = T.weighted_n_node_samples[0]
    R_node = T.impurity * T.weighted_n_node_samples / n_total          # R(t) nếu t thành lá
    R_sub = np.zeros(T.node_count)
    n_leaves = np.zeros(T.node_count)
    for node in range(T.node_count - 1, -1, -1):                     # con luôn có id lớn hơn cha
        left, right = T.children_left[node], T.children_right[node]
        ### START CODE HERE ### (≈ 5 dòng)
        # lá (left == -1): R_sub = R_node, n_leaves = 1; node trong: cộng từ hai con
        pass
        ### END CODE HERE ###
    internal = T.children_left != -1
    g = (R_node[internal] - R_sub[internal]) / (n_leaves[internal] - 1)
    return g.min()

In [ ]:
small_tree = DecisionTreeRegressor(max_depth=4, random_state=0).fit(Xh_tr, yh_tr)
alpha_ours = weakest_link_alpha(small_tree)
alphas_sk = DecisionTreeRegressor(max_depth=4, random_state=0).cost_complexity_pruning_path(Xh_tr, yh_tr).ccp_alphas
print(f"weakest link (scratch) = {alpha_ours:.6f} | alpha đầu tiên > 0 của sklearn = {alphas_sk[alphas_sk > 0][0]:.6f}")
check_close(alpha_ours, alphas_sk[alphas_sk > 0][0], tol=1e-9, message="khớp sklearn")